In [19]:
import os
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import faiss
from transformers import AutoTokenizer, AutoModel
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from dotenv import load_dotenv

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print(f"Project root: {ROOT}")

os.makedirs(os.path.join(ROOT, "rag_artifacts_v3"), exist_ok=True)
print("✓ rag_artifacts/ created")

load_dotenv(os.path.join(ROOT, "secrets.env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "⚠ HF_TOKEN missing or invalid"
print("✓ HF_TOKEN loaded")


Project root: /Users/avani/IdeaProjects/customer-support-management
✓ rag_artifacts/ created
✓ HF_TOKEN loaded


In [20]:
MODEL_ID = "Nethra19/multitask-ticket-model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
encoder   = AutoModel.from_pretrained(MODEL_ID, token=HF_TOKEN)
encoder   = encoder.to(device)
encoder.eval()

print(f"✓ Encoder loaded : {encoder.__class__.__name__}")
print(f"  Hidden size    : {encoder.config.hidden_size}")
print(f"  Params         : {sum(p.numel() for p in encoder.parameters()):,}")


In [21]:
class FineTunedTicketEmbedder:
    """
    Wraps the fine-tuned DistilBERT encoder.
    CLS token → L2 normalised float32 embedding.
    Same representation the classifier heads were trained on.
    """
    def __init__(self, encoder, tokenizer, device, batch_size=32):
        self.encoder    = encoder
        self.tokenizer  = tokenizer
        self.device     = device
        self.batch_size = batch_size

    def _encode(self, texts: list) -> np.ndarray:
        all_embs = []
        for i in range(0, len(texts), self.batch_size):
            batch  = texts[i : i + self.batch_size]
            inputs = self.tokenizer(
                batch, return_tensors="pt",
                truncation=True, padding=True, max_length=256,
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()
                      if k != "token_type_ids"}
            with torch.no_grad():
                out  = self.encoder(**inputs)
                cls  = out.last_hidden_state[:, 0, :]        # [CLS] token
                cls  = F.normalize(cls, p=2, dim=-1)          # L2 normalise
            all_embs.append(cls.cpu().numpy())
        return np.vstack(all_embs).astype("float32")

    def embed_documents(self, texts: list) -> np.ndarray:
        return self._encode(texts)

    def embed_query(self, text: str) -> np.ndarray:
        return self._encode([text])[0]


embedder = FineTunedTicketEmbedder(encoder, tokenizer, device)

# Sanity check
test = embedder.embed_query("production server crashed")
print(f"✓ Embedder ready")
print(f"  Dim  : {test.shape[0]}")
print(f"  Norm : {np.linalg.norm(test):.4f}  ← should be 1.0")


✓ Embedder ready
  Dim  : 768
  Norm : 1.0000  ← should be 1.0


In [22]:
COMPLIANCE_DIR = os.path.join(ROOT, "compliance_docs")

DEPT_FILE_MAP = {
    "Billing_and_Payments.txt":              "Billing and Payments",
    "Customer_Service.txt":                  "Customer Service",
    "General_Inquiry.txt":                   "General Inquiry",
    "Human_Resources.txt":                   "Human Resources",
    "Technical & IT Support.txt":            "Technical & IT Support",
    "Returns_and_Exchanges.txt":             "Returns and Exchanges",
    "Sales_and_Pre-Sales.txt":               "Sales and Pre-Sales",
    "Service_Outages_and_Maintenance.txt":   "Service Outages and Maintenance",
    "Priority_Escalation_Criteria.txt":      "PRIORITY",
}


def split_into_sections(text, dept_label):
    section_patterns = [
        ("OVERVIEW",  r"OVERVIEW\n-+\n"),
        ("HIGH",      r"PRIORITY: HIGH\n-+\n"),
        ("MEDIUM",    r"PRIORITY: MEDIUM\n-+\n"),
        ("LOW",       r"PRIORITY: LOW\n-+\n"),
        ("ROUTING",   r"ROUTING DECISION GUIDANCE\n-+\n"),
    ]
    positions = []
    for name, pattern in section_patterns:
        for m in re.finditer(pattern, text):
            positions.append((m.start(), m.end(), name))
    positions.sort()

    chunks = []
    for i, (start, end, name) in enumerate(positions):
        next_start = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        body = text[end:next_start].strip()
        if not body:
            continue
        priority_hint = name if name in ("HIGH", "MEDIUM", "LOW") else "GENERAL"
        prefix = (
            f"[DEPARTMENT: {dept_label}] "
            f"[PRIORITY_SECTION: {priority_hint}] "
            f"[SECTION: {name}]\n\n"
        )
        chunks.append({
            "text":          prefix + body,
            "raw_body":      body,
            "dept":          dept_label,
            "priority_hint": priority_hint,
            "section":       name,
            "chunk_type":    "section",
        })
    return chunks


def sliding_window_chunks(text, dept_label, window=400, step=200):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        body = " ".join(words[i : i + window])
        prefix = (
            f"[DEPARTMENT: {dept_label}] "
            f"[PRIORITY_SECTION: GENERAL] "
            f"[SECTION: SLIDING_WINDOW] "
            f"[OFFSET: {i}]\n\n"
        )
        chunks.append({
            "text":          prefix + body,
            "raw_body":      body,
            "dept":          dept_label,
            "priority_hint": "GENERAL",
            "section":       f"SLIDING_WINDOW_{i}",
            "chunk_type":    "sliding",
        })
        if i + window >= len(words):
            break
        i += step
    return chunks


print("✓ Chunking functions defined")


✓ Chunking functions defined


In [23]:
all_chunks = []
for fname, dept_label in DEPT_FILE_MAP.items():
    fpath = os.path.join(COMPLIANCE_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  ⚠  MISSING: {fname}")
        continue
    with open(fpath, encoding="utf-8") as f:
        text = f.read()

    sec_chunks = split_into_sections(text, dept_label)
    slw_chunks = sliding_window_chunks(text, dept_label, window=400, step=200)
    all_chunks.extend(sec_chunks)
    all_chunks.extend(slw_chunks)

    print(f"  ✓  {fname:<45}  section:{len(sec_chunks)}  sliding:{len(slw_chunks)}")

print(f"\n✓ Total chunks to index: {len(all_chunks)}")
print(f"  Dept chunks    : {len([c for c in all_chunks if c['dept'] != 'PRIORITY'])}")
print(f"  Priority chunks: {len([c for c in all_chunks if c['dept'] == 'PRIORITY'])}")


  ✓  Billing_and_Payments.txt                       section:2  sliding:2
  ✓  Customer_Service.txt                           section:2  sliding:2
  ✓  General_Inquiry.txt                            section:2  sliding:2
  ✓  Human_Resources.txt                            section:2  sliding:2
  ✓  Technical & IT Support.txt                     section:2  sliding:3
  ✓  Returns_and_Exchanges.txt                      section:2  sliding:3
  ✓  Sales_and_Pre-Sales.txt                        section:2  sliding:3
  ✓  Service_Outages_and_Maintenance.txt            section:2  sliding:3
  ✓  Priority_Escalation_Criteria.txt               section:4  sliding:4

✓ Total chunks to index: 44
  Dept chunks    : 36
  Priority chunks: 8


In [24]:
# ── Split chunks before indexing ─────────────────────────────────────
dept_chunks     = [c for c in all_chunks if c["dept"] != "PRIORITY"]
priority_chunks = [
    c for c in all_chunks
    if c["dept"] == "PRIORITY" and c["priority_hint"] in ("HIGH", "MEDIUM", "LOW")
]

print(f"Dept chunks     : {len(dept_chunks)}")
print(f"Priority chunks : {len(priority_chunks)}")
print(f"  Sections      : {[c['section'] for c in priority_chunks]}")

# ── Embed dept chunks → HNSW index ───────────────────────────────────
print(f"\nEmbedding {len(dept_chunks)} dept chunks...")
dept_texts      = [c["text"] for c in dept_chunks]
dept_embeddings = embedder.embed_documents(dept_texts)
print(f"✓ Dept embeddings shape: {dept_embeddings.shape}")

dim        = dept_embeddings.shape[1]
hnsw_index = faiss.IndexHNSWFlat(dim, 32)
hnsw_index.hnsw.efConstruction = 200
hnsw_index.hnsw.efSearch        = 64
hnsw_index.add(dept_embeddings)

print(f"✓ HNSW dept index built — {hnsw_index.ntotal} vectors")

# ── Embed priority chunks → flat exact index ─────────────────────────
print(f"\nEmbedding {len(priority_chunks)} priority chunks...")
priority_texts      = [c["text"] for c in priority_chunks]
priority_embeddings = embedder.embed_documents(priority_texts)

priority_index = faiss.IndexFlatIP(dim)
priority_index.add(priority_embeddings.astype("float32"))

print(f"✓ Priority flat index built — {priority_index.ntotal} vectors")

✓ Dept embeddings shape: (36, 768)
✓ HNSW dept index built — 36 vectors

Embedding 3 priority chunks...
✓ Priority flat index built — 3 vectors


In [25]:
def tokenize_for_bm25(text: str) -> list:
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

tokenized_corpus = [tokenize_for_bm25(c["text"]) for c in dept_chunks]
bm25             = BM25Okapi(tokenized_corpus)

print(f"✓ BM25 index built over {len(tokenized_corpus)} dept documents")


✓ BM25 index built over 36 dept documents


In [26]:
ARTIFACTS_DIR = os.path.join(ROOT, "rag_artifacts_v3")

# 1. FAISS HNSW index  (dept chunks only — unchanged)
faiss_path = os.path.join(ARTIFACTS_DIR, "rag_compliance_index.faiss")
faiss.write_index(hnsw_index, faiss_path)
print(f"✓ FAISS saved           → rag_artifacts_v3/rag_compliance_index.faiss")

# 2. BM25 + tokenized corpus  (dept chunks only — unchanged)
bm25_path = os.path.join(ARTIFACTS_DIR, "rag_bm25_index.pkl")
with open(bm25_path, "wb") as f:
    pickle.dump({"bm25": bm25, "tokenized_corpus": tokenized_corpus}, f)
print(f"✓ BM25 saved            → rag_artifacts_v3/rag_bm25_index.pkl")

# 3. Dept chunk metadata  (unchanged)
meta_path = os.path.join(ARTIFACTS_DIR, "rag_compliance_metadata.pkl")
with open(meta_path, "wb") as f:
    pickle.dump(dept_chunks, f)
print(f"✓ Dept metadata saved   → rag_artifacts_v3/rag_compliance_metadata.pkl")

# 4. Tokenizer helper  (unchanged)
helper_path = os.path.join(ARTIFACTS_DIR, "retrieval_helpers.pkl")
with open(helper_path, "wb") as f:
    pickle.dump({"tokenize_for_bm25": tokenize_for_bm25}, f)
print(f"✓ Helpers saved         → rag_artifacts_v3/retrieval_helpers.pkl")

# 5. Priority FAISS flat index  ← NEW
priority_faiss_path = os.path.join(ARTIFACTS_DIR, "rag_priority_index.faiss")
faiss.write_index(priority_index, priority_faiss_path)
print(f"✓ Priority FAISS saved  → rag_artifacts_v3/rag_priority_index.faiss")

# 6. Priority chunk metadata  ← NEW
priority_meta_path = os.path.join(ARTIFACTS_DIR, "rag_priority_metadata.pkl")
with open(priority_meta_path, "wb") as f:
    pickle.dump(priority_chunks, f)
print(f"✓ Priority metadata     → rag_artifacts_v3/rag_priority_metadata.pkl")

print(f"\n✓ All 6 artifacts saved to rag_artifacts_v3/")

✓ FAISS saved           → rag_artifacts_v3/rag_compliance_index.faiss
✓ BM25 saved            → rag_artifacts_v3/rag_bm25_index.pkl
✓ Dept metadata saved   → rag_artifacts_v3/rag_compliance_metadata.pkl
✓ Helpers saved         → rag_artifacts_v3/retrieval_helpers.pkl
✓ Priority FAISS saved  → rag_artifacts_v3/rag_priority_index.faiss
✓ Priority metadata     → rag_artifacts_v3/rag_priority_metadata.pkl

✓ All 6 artifacts saved to rag_artifacts_v3/


In [27]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✓ CrossEncoder reranker loaded\n")


def hybrid_retrieve(query: str, top_k_dense=10, top_k_bm25=10, top_n_final=3):
    """
    HNSW dense + BM25 → RRF fusion → CrossEncoder rerank.
    Returns top_n_final chunks with scores.
    """
    # 1. Dense retrieval
    q_emb = embedder.embed_query(query).reshape(1, -1)
    _, dense_ids = hnsw_index.search(q_emb, top_k_dense)
    dense_ids = dense_ids[0].tolist()

    # 2. BM25 retrieval
    q_tokens  = tokenize_for_bm25(query)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_ids    = np.argsort(bm25_scores)[::-1][:top_k_bm25].tolist()

    # 3. RRF fusion (k=60 is the standard default from the lecture)
    rrf = {}
    for rank, idx in enumerate(dense_ids):
        rrf[idx] = rrf.get(idx, 0) + 1.0 / (60 + rank + 1)
    for rank, idx in enumerate(bm25_ids):
        rrf[idx] = rrf.get(idx, 0) + 1.0 / (60 + rank + 1)

    candidate_ids   = [idx for idx, _ in sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:20]]
    candidate_texts = [dept_chunks[i]["text"] for i in candidate_ids]

    # 4. CrossEncoder rerank
    pairs     = [[query, t] for t in candidate_texts]
    ce_scores = cross_encoder.predict(pairs)
    ranked    = sorted(zip(candidate_ids, ce_scores), key=lambda x: x[1], reverse=True)

    return [
        {"chunk": dept_chunks[idx], "ce_score": float(score)}
        for idx, score in ranked[:top_n_final]
    ]


# ── Test queries covering all 8 departments ──────────────────────────
TEST_QUERIES = [
    ("Technical & IT Support",          "Production server is down, database connection failing, all users affected"),
    ("Billing and Payments",            "I was charged twice for my subscription this month, need a refund"),
    ("Returns and Exchanges",           "I received a damaged laptop, want to return it and get a replacement"),
    ("Service Outages and Maintenance", "The entire platform has been unreachable for 3 hours for all users globally"),
    ("Sales and Pre-Sales",             "I want to know pricing for your enterprise plan before we sign"),
    ("Human Resources",                 "My payroll is wrong this month and payday is tomorrow"),
    ("General Inquiry",                 "What are your support hours and how do I contact the team"),
    ("Customer Service",                "I am very unhappy with how my last complaint was handled, I want to escalate"),
]

print("=" * 75)
print("SMOKE TEST — HYBRID RETRIEVAL (HNSW + BM25 + RRF + CrossEncoder)")
print("=" * 75)

correct = 0
for expected_dept, query in TEST_QUERIES:
    results   = hybrid_retrieve(query, top_n_final=3)
    top_dept  = results[0]["chunk"]["dept"]
    top_score = results[0]["ce_score"]
    match     = "✓" if top_dept == expected_dept else "✗"
    if top_dept == expected_dept:
        correct += 1
    print(f"\n{match}  Expected : {expected_dept}")
    print(f"   Got      : {top_dept}  (score={top_score:.3f})")
    print(f"   Query    : {query[:65]}")

print(f"\n{'=' * 75}")
print(f"Retrieval accuracy on smoke test: {correct}/{len(TEST_QUERIES)}")
print("=" * 75)

# ── Formal retrieval evaluation ─────────────────────────
print("\n" + "=" * 75)
print("RETRIEVAL EVALUATION — Context Precision & Recall")
print("=" * 75)

for expected_dept, query in TEST_QUERIES:
    results    = hybrid_retrieve(query, top_n_final=5)
    retrieved  = [r["chunk"]["dept"] for r in results]

    # Context Precision: how many retrieved chunks match expected dept
    precision  = sum(1 for d in retrieved if d == expected_dept) / len(retrieved)

    # Context Recall: did the expected dept appear at all in top-5?
    recall     = 1.0 if expected_dept in retrieved else 0.0

    print(f"  {expected_dept:<42}  P={precision:.2f}  R={recall:.1f}")



# ── Priority retrieval smoke test ────────────────────────────
def retrieve_priority(query, k=1):
    q_emb = embedder.embed_query(query).reshape(1, -1)
    _, ids = priority_index.search(q_emb, k)
    cand_ids = ids[0].tolist()
    pairs = [[query, priority_chunks[i]["text"]] for i in cand_ids]
    ce_scores = cross_encoder.predict(pairs)
    best = int(np.argmax(ce_scores))
    chunk = priority_chunks[cand_ids[best]]
    return chunk["section"], float(ce_scores[best])

PRIORITY_QUERIES = [
    ("HIGH",   "Production database is down, revenue loss, all customers affected"),
    ("HIGH",   "Security breach detected, unauthorised access to customer data"),
    ("MEDIUM", "Single user cannot access their account, partial workaround available"),
    ("LOW",    "General question about support hours, no urgency"),
]

print("\n" + "=" * 75)
print("SMOKE TEST — PRIORITY RETRIEVAL")
print("=" * 75)

correct = 0
for expected, query in PRIORITY_QUERIES:
    section, score = retrieve_priority(query)
    match = "✓" if section == expected else "✗"
    if section == expected:
        correct += 1
    print(f"\n{match}  Expected : {expected}")
    print(f"   Got      : {section}  (CE={score:.3f})")
    print(f"   Query    : {query[:65]}")

print(f"\n{'=' * 75}")
print(f"Priority retrieval accuracy: {correct}/{len(PRIORITY_QUERIES)}")
print("=" * 75)


  Customer Service                            P=0.60  R=1.0

SMOKE TEST — PRIORITY RETRIEVAL

✓  Expected : HIGH
   Got      : HIGH  (CE=-0.139)
   Query    : Production database is down, revenue loss, all customers affected

✗  Expected : HIGH
   Got      : MEDIUM  (CE=-6.386)
   Query    : Security breach detected, unauthorised access to customer data

✗  Expected : MEDIUM
   Got      : HIGH  (CE=-2.003)
   Query    : Single user cannot access their account, partial workaround avail

✗  Expected : LOW
   Got      : HIGH  (CE=-4.285)
   Query    : General question about support hours, no urgency

Priority retrieval accuracy: 1/4


In [28]:
from huggingface_hub import HfApi, login, create_repo
import time
HF_TOKEN_AVANI = os.getenv("HF_TOKEN_AVANI")
login(token=HF_TOKEN_AVANI)
api  = HfApi()
REPO = "avani1010/new_index"

# ── Step 1: Create repo if it doesn't exist ──────────────────
try:
    create_repo(
        repo_id=REPO,
        repo_type="model",
        private=False,
        token=HF_TOKEN,
        exist_ok=True,      # won't error if already exists
    )
    print(f"✓ Repo ready: https://huggingface.co/{REPO}")
except Exception as e:
    print(f"⚠ Repo creation note: {e}")

# ── Step 2: Upload with retry ─────────────────────────────────
FILES = [
    (os.path.join(ROOT, "rag_artifacts_v3/rag_compliance_index.faiss"),  "rag_compliance_index.faiss"),
    (os.path.join(ROOT, "rag_artifacts_v3/rag_bm25_index.pkl"),          "rag_bm25_index.pkl"),
    (os.path.join(ROOT, "rag_artifacts_v3/rag_compliance_metadata.pkl"), "rag_compliance_metadata.pkl"),
    (os.path.join(ROOT, "rag_artifacts_v3/rag_priority_index.faiss"),    "rag_priority_index.faiss"),    # ← NEW
    (os.path.join(ROOT, "rag_artifacts_v3/rag_priority_metadata.pkl"),   "rag_priority_metadata.pkl"),   # ← NEW
]

for local_path, repo_path in FILES:
    for attempt in range(1, 4):          # 3 attempts max
        try:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=REPO,
                repo_type="model",
                token=HF_TOKEN,
                commit_message=f"Hybrid RAG artifacts [{repo_path}]",
            )
            print(f"  ✓ Uploaded → {repo_path}")
            break
        except Exception as e:
            print(f"  ⚠ Attempt {attempt}/3 failed for {repo_path}: {type(e).__name__}")
            print(e)
            if attempt < 3:
                time.sleep(5 * attempt)   # wait 5s, then 10s before retrying
            else:
                print(f"  ✗ Giving up on {repo_path} — upload manually if needed")

print(f"\n✓ Done. Check: https://huggingface.co/{REPO}")

  ⚠ Attempt 3/3 failed for rag_priority_metadata.pkl: HfHubHTTPError
(Request ID: Root=1-69d198dc-16cee8bc6b36bd064a4b4d94;fb58e245-3b59-4f84-86e0-8d4a8c81856c)

403 Forbidden: You have read access but not the required permissions for this operation.
Cannot access content at: https://huggingface.co/avani1010/new_index.git/info/lfs/objects/batch.
Make sure your token has the correct permissions.
  ✗ Giving up on rag_priority_metadata.pkl — upload manually if needed

✓ Done. Check: https://huggingface.co/avani1010/new_index
